# Harry: compare first, then plot

This notebook runs FieldMatch comparisons and makes reusable Matplotlib figures.
Edit the settings cell and run the notebook from top to bottom. The accompanying
`harry_campaign.yaml` declares the datasets and experiments. No data are downloaded.

The first run reads the GRIB files and can take several minutes. To redraw existing
results, set `RUN_COMPARISONS = False`; provenance checks reject results whose campaign
or source files have changed. Install the `notebook` extra to use Jupyter.

**Scientific choices:** BA08 comparisons use independent Hs matching, followed by an
explicit common-observation intersection. Grid maps use exact common valid times,
the declared reference grid and a common finite mask at each time. Forecast maps
also require the same initialization. Analysis is a reference, not truth.

In [ ]:
from pathlib import Path
import os

# Change these settings for your installation and the figure you want.
DATA_ROOT = Path(os.environ.get("FIELDMATCH_DATA_ROOT", "../../harry_storm/data")).resolve()
OUTPUT = Path(os.environ.get("FIELDMATCH_OUTPUT", "results/harry")).resolve()
CONFIG = Path("harry_campaign.yaml")
RUN_COMPARISONS = True
MAP_TIME = "2026-01-20T18:00"  # Must exist exactly in the saved comparison.
HS_LIMITS = (0, 10)             # metres; shared between reference and candidate.
DIFFERENCE_LIMIT = 2            # metres; symmetric difference colour scale.

# More scientific settings (variables, interpolation, tolerances, forecast selection)
# are grouped visibly in CONFIG. Both map examples below use Hs.


In [ ]:
import json
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from fieldmatch.campaign import load_campaign
from fieldmatch.comparison import resolve_comparisons, run_comparisons
from fieldmatch.results import open_result
from fieldmatch.common import common_sample
from fieldmatch.grids import grid_stats
from fieldmatch.pairstats import pair_stats
from fieldmatch.plotting import comparison_panels, time_series, scatter, save_figure

OUTPUT.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load(CONFIG.read_text())
config["data_root"] = str(DATA_ROOT)
config["outdir"] = str(OUTPUT / "comparisons")
campaign_file = OUTPUT / "campaign.yaml"
campaign_file.write_text(yaml.safe_dump(config, sort_keys=False))
campaign = load_campaign(campaign_file)

# Inspect these resolved choices before interpreting the figures.
summary = []
for name in campaign.comparisons:
    for spec in resolve_comparisons(campaign, name):
        summary.append(dict(comparison=name, variable=spec["variable"],
            time_method=spec.get("time_method", spec["matching"].get("time_method")),
            time_basis=spec.get("time_basis", "observation valid time"),
            grid=spec.get("target_grid", "observation positions"),
            space_method=spec["matching"]["space_method"]))
display(pd.DataFrame(summary))


In [ ]:
if RUN_COMPARISONS:
    specs = [spec for name in campaign.comparisons
             for spec in resolve_comparisons(campaign, name)]
    completed, failed = run_comparisons(campaign, specs, formats=("netcdf", "csv"))
    if failed:
        raise RuntimeError(failed)

def result(name, variable="hs"):
    """Find the named result without depending on its filename spelling."""
    matches = []
    for path in campaign.outdir.glob("*.manifest.json"):
        manifest = json.loads(path.read_text())
        effective = manifest.get("effective", {})
        if effective.get("comparison") == name and effective.get("variable") == variable:
            if manifest["status"] != "complete":
                raise RuntimeError(manifest.get("reason", manifest["status"]))
            matches.append(manifest["outputs"]["netcdf"])
    if len(matches) != 1:
        raise ValueError(f"Expected one complete {name}/{variable} result, found {len(matches)}")
    return open_result(matches[0])


## BA08: common-observation time series and scatter

The intersection happens explicitly here, before plotting. The table reports how
many records each model loses to the common sample. Points are drawn at observation
times; the original `model_time`, `dt`, initialization and lead remain in each result.
The default plot uses markers so it does not imply continuous model evolution.

In [ ]:
pairs = {label: result(name) for label, name in {
    "Analysis": "buoy_analysis", "Hindcast": "buoy_hindcast", "ERA5": "buoy_era5"
}.items()}
aligned, counts = common_sample(pairs)
display(pd.DataFrame(counts).T)
fig, ax = time_series(aligned, title="BA08 — identical observations, 18–22 January 2026")
save_figure(fig, OUTPUT / "ba08_time_series.png")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, (label, table) in zip(axes, aligned.items()):
    scatter(table, ax=ax, title=label)
    ax.set(xlim=(0,10), ylim=(0,10))
save_figure(fig, OUTPUT / "ba08_scatter.png")
plt.show()
display(pd.DataFrame({label: pair_stats(ds.hs.values, ds.model_hs.values)
                      for label, ds in aligned.items()}).T)


## Model fields: reference, candidate and signed difference

The first figure uses the analysis grid; interpolating ERA5 onto this finer grid
does **not** increase its physical resolution. Only cells finite in both products
are shown. The second compares forecasts from the same initialization at exact
shared times. Changing `MAP_TIME` never selects a nearby timestamp silently.
These are regional geographic plots with a latitude aspect correction, not a
cartographic projection. Grey/blank areas have no common valid value.

In [ ]:
for name in ["analysis_era5", "forecast18_models"]:
    grid = result(name)
    fig, axes = comparison_panels(grid, MAP_TIME, clim=HS_LIMITS,
                                  difference_limit=DIFFERENCE_LIMIT)
    save_figure(fig, OUTPUT / f"{name}_panels.png")
    plt.show()
    summary = grid_stats(grid)
    display(summary[["mean_difference", "rms_difference", "valid_area_fraction", "n_common"]]
            .sel(time=[MAP_TIME]).to_dataframe())


## Inspect a decision or change an experiment

- Open `harry_campaign.yaml` to change the station, region, dates or model selections.
- Under an observation variable, use `tolerance_minutes: 0` for exact timestamps.
- Under a grid variable, select `space_method: nearest` or `bilinear`; grid times stay exact.
- `reference` sets the grid and the subtracted field. `model` is the candidate.
- Use `valid_time` for reference consistency, `same_init` for matching forecast runs,
  or `same_lead` to require matching finite leads at common valid times.
- Different physical quantities need compatible source names and units. Do not map
  ECMWF `mwp` to buoy `tm01` or `tm02`, or neutral forcing winds to atmospheric winds.
- Adding another buoy comparison requires its source coordinates to be verified.

Every saved figure has a `.figure.json` sidecar with the input-result hashes,
comparison settings and plot limits. The CSV for a grid experiment contains
per-time area-weighted difference summaries; its NetCDF contains the full fields.
Use ordinary Matplotlib calls to adjust labels, panel layouts or output formats.
No matching or interpolation happens inside the plotting module.